In [1]:
from sedona.spark import SedonaContext
import os
from sedona.spark.maps.SedonaKepler import SedonaKepler

In [2]:
%%capture
bucket_name = os.environ.get("SEDONA_SOURCE_BUCKET", "apache-sedona-book")

config = SedonaContext.builder()

sedona = SedonaContext.create(config.getOrCreate())

sedona.sparkContext.setLogLevel("ERROR")

sc = sedona.sparkContext

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/09/05 21:07:53 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/09/05 21:07:57 WARN UDTRegistration: Cannot register UDT for org.geotools.coverage.grid.GridCoverage2D, which is already registered.
25/09/05 21:07:57 WARN SimpleFunctionRegistry: The function rs_union_aggr replaced a previously registered function.
25/09/05 21:07:57 WARN UDTRegistration: Cannot register UDT for org.locationtech.jts.geom.Geometry, which is already registered.
25/09/05 21:07:57 WARN UDTRegistration: Cannot register UDT for org.apache.sedona.common.S2Geography.Geography, which is already registered.
25/09/05 21:07:57 WARN UDTRegistration: Cannot register UDT for org.locationtech.jts.index.SpatialIndex, which is already registered.
25/09/05 21:07:57 WARN SimpleFunctionRegistry: The function st_envelop

In [ ]:
spatial_df = sedona.\
    read.\
    format("geoparquet").\
    load(f"s3a://{bucket_name}/source_data/arizona_buildings")

roads_df = sedona.\
    read.\
    format("geoparquet").\
    load(f"s3a://{bucket_name}/source_data/arizona_roads")

# Kepler GL

In [ ]:
map_view = SedonaKepler.create_map(spatial_df, "Sedona Buildings")

SedonaKepler.add_df(map_view, roads_df, "Sedona Roads")

In [ ]:
map_view

# GeoPandas

In [ ]:
!pip install contextily

In [ ]:
import geopandas as gpd
import pyspark.sql.functions as f
import contextily as cx
from sedona.spark import dataframe_to_arrow

gdf = gpd.GeoDataFrame.from_arrow(dataframe_to_arrow(spatial_df))

In [ ]:
gdf = gdf.set_crs(crs="epsg:4326")

In [ ]:
ax = gdf.to_crs(epsg=3857)\
    .plot(
        column="type",
        figsize=(10, 10),
        missing_kwds={
                "color": "lightgrey",
                "edgecolor": "red",
                "label": "Missing values",
            },
        legend=True,
        legend_kwds={"title": "Sedona buildings"},
    )



cx.add_basemap(ax)

# Pydeck

In [ ]:
from sedona.spark.maps.SedonaPyDeck import SedonaPyDeck



In [ ]:
crimes = sedona.read.format("geoparquet")\
    .load(f"s3a://{bucket_name}/source_data/crimes")\
    .selectExpr("id", "geom")

In [ ]:
SedonaPyDeck.create_heatmap(
    crimes,
    aggregation="SUM",
    map_style="light"
)


# raster viz

In [ ]:
from sedona.spark.raster_utils.SedonaUtils import SedonaUtils


In [23]:
raster_df = sedona.read.format("binaryFile")\
    .load(f"s3a://{bucket_name}/source_data/raster_viz")\
    .selectExpr("RS_FromGeoTiff(content)")

In [ ]:
SedonaUtils.display_image(raster_df)